# Pairs Trading Strategy Demo

This notebook demonstrates the pairs trading implementation integrated into the ML4T trading system.

## Overview

Pairs trading is a market-neutral statistical arbitrage strategy that:
1. Identifies two cointegrated securities
2. Monitors their spread for mean reversion opportunities
3. Takes offsetting positions when the spread diverges
4. Profits when the spread reverts to its historical mean

## Key Features

- **Statistical Testing**: Engle-Granger cointegration testing
- **Signal Generation**: Z-score based entry/exit signals
- **Risk Management**: Stop-loss and position sizing controls
- **Performance Evaluation**: Comprehensive backtesting framework
- **Integration**: Seamless integration with existing trading system

In [ ]:
# Import required libraries
import sys
import os
sys.path.append('trading_system')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Set up plotting
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
%matplotlib inline

## 1. Import Pairs Trading Components

In [ ]:
# Import pairs trading strategy components
from strategies.pairs_trading import PairsStrategy, PairCandidate, create_pairs_strategy
from backtesting.pairs_backtest import PairsBacktester, BacktestConfig, run_pairs_backtest, generate_backtest_report
from tools.pairs_trading_tools import (
    PairAnalysisTool, SpreadAnalysisTool, PairsRiskAssessmentTool,
    create_pairs_trading_tools
)
from agents.pairs_trading_agent import PairsTradingAgent

print("✅ Pairs trading components imported successfully")

## 2. Generate Sample Data for Demo

Since we don't want to depend on external APIs for this demo, we'll create synthetic cointegrated pairs data.

In [ ]:
def generate_cointegrated_pair(n_days=500, start_price_a=100, start_price_b=50, beta=1.8, noise_std=0.02):
    """
    Generate synthetic cointegrated pair data for demonstration
    """
    np.random.seed(42)  # For reproducible results
    
    # Generate common trend
    common_trend = np.cumsum(np.random.normal(0, 0.01, n_days))
    
    # Generate mean-reverting spread
    spread = np.zeros(n_days)
    spread[0] = 0
    
    for i in range(1, n_days):
        # AR(1) process with mean reversion
        spread[i] = 0.95 * spread[i-1] + np.random.normal(0, 0.5)
    
    # Generate prices
    price_b = start_price_b * np.exp(common_trend + np.random.normal(0, noise_std, n_days))
    price_a = start_price_a * np.exp(common_trend) + beta * (price_b - price_b[0]) + spread
    
    # Ensure positive prices
    price_a = np.maximum(price_a, 1.0)
    price_b = np.maximum(price_b, 1.0)
    
    # Create date index
    dates = pd.date_range(start='2023-01-01', periods=n_days, freq='D')
    
    return pd.Series(price_a, index=dates), pd.Series(price_b, index=dates)

# Generate sample data for multiple pairs
print("Generating synthetic cointegrated pairs data...")

# Generate 6 synthetic stocks with 3 cointegrated pairs
stock_a, stock_b = generate_cointegrated_pair(500, 100, 50, 1.8, 0.02)  # TECH_A - TECH_B
stock_c, stock_d = generate_cointegrated_pair(500, 80, 60, 1.2, 0.015)  # ENERGY_A - ENERGY_B  
stock_e, stock_f = generate_cointegrated_pair(500, 120, 90, 1.5, 0.025) # FINANCE_A - FINANCE_B

# Create price data dictionary
price_data = {
    'TECH_A': stock_a,
    'TECH_B': stock_b,
    'ENERGY_A': stock_c,
    'ENERGY_B': stock_d,
    'FINANCE_A': stock_e,
    'FINANCE_B': stock_f
}

print(f"✅ Generated {len(price_data)} synthetic stocks with {len(price_data['TECH_A'])} days of data")
print(f"Date range: {price_data['TECH_A'].index[0].strftime('%Y-%m-%d')} to {price_data['TECH_A'].index[-1].strftime('%Y-%m-%d')}")

## 3. Visualize Sample Data

In [ ]:
# Plot the synthetic pairs
fig, axes = plt.subplots(3, 2, figsize=(15, 12))
fig.suptitle('Synthetic Cointegrated Pairs Data', fontsize=16, fontweight='bold')

pairs = [('TECH_A', 'TECH_B'), ('ENERGY_A', 'ENERGY_B'), ('FINANCE_A', 'FINANCE_B')]

for i, (symbol_a, symbol_b) in enumerate(pairs):
    # Plot normalized prices
    axes[i, 0].plot(price_data[symbol_a] / price_data[symbol_a].iloc[0], label=symbol_a, linewidth=2)
    axes[i, 0].plot(price_data[symbol_b] / price_data[symbol_b].iloc[0], label=symbol_b, linewidth=2)
    axes[i, 0].set_title(f'{symbol_a} vs {symbol_b} (Normalized)')
    axes[i, 0].legend()
    axes[i, 0].grid(True, alpha=0.3)
    
    # Plot spread
    spread = price_data[symbol_a] - 1.5 * price_data[symbol_b]  # Approximate hedge ratio
    axes[i, 1].plot(spread, color='red', linewidth=2)
    axes[i, 1].axhline(y=spread.mean(), color='black', linestyle='--', alpha=0.7, label='Mean')
    axes[i, 1].axhline(y=spread.mean() + 2*spread.std(), color='green', linestyle='--', alpha=0.7, label='+2σ')
    axes[i, 1].axhline(y=spread.mean() - 2*spread.std(), color='green', linestyle='--', alpha=0.7, label='-2σ')
    axes[i, 1].set_title(f'{symbol_a}-{symbol_b} Spread')
    axes[i, 1].legend()
    axes[i, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Initialize Pairs Trading Strategy

In [ ]:
# Initialize pairs trading strategy
strategy_config = {
    'lookback_period': 252,      # 1 year for pair selection
    'signal_window': 20,         # 20-day rolling window for signals
    'entry_zscore': 2.0,         # Enter when |z-score| >= 2.0
    'exit_zscore': 0.0,          # Exit when z-score crosses zero
    'stop_loss_zscore': 3.0,     # Stop loss at |z-score| >= 3.0
    'max_positions': 3,          # Maximum 3 active pairs
    'position_size': 10000.0,    # $10k per leg
    'correlation_threshold': 0.7, # Minimum correlation
    'cointegration_pvalue': 0.05  # Maximum p-value for cointegration
}

pairs_strategy = create_pairs_strategy(strategy_config)

print("✅ Pairs trading strategy initialized")
print(f"Strategy parameters:")
for key, value in strategy_config.items():
    print(f"  {key}: {value}")

## 5. Find Cointegrated Pairs

In [ ]:
# Find cointegrated pairs from our synthetic data
symbols = list(price_data.keys())
print(f"Analyzing {len(symbols)} symbols for cointegrated pairs...")
print(f"Symbols: {symbols}")

# Find pairs using the strategy
found_pairs = pairs_strategy.find_pairs(symbols, price_data)

print(f"\n✅ Found {len(found_pairs)} cointegrated pairs:")
print("\nPair Analysis Results:")
print("=" * 80)
print(f"{'Pair':<20} {'Correlation':<12} {'P-Value':<10} {'Half-Life':<12} {'Score':<8}")
print("=" * 80)

for pair in found_pairs:
    pair_name = f"{pair.symbol_a}-{pair.symbol_b}"
    print(f"{pair_name:<20} {pair.correlation:<12.3f} {pair.p_value:<10.4f} {pair.half_life:<12.1f} {pair.score:<8.3f}")

if found_pairs:
    print(f"\nTop pair: {found_pairs[0].symbol_a}-{found_pairs[0].symbol_b} (score: {found_pairs[0].score:.3f})")
else:
    print("No pairs found meeting the criteria.")

## 6. Analyze Specific Pair Spread

In [ ]:
if found_pairs:
    # Analyze the top pair in detail
    top_pair = found_pairs[0]
    
    print(f"Detailed Analysis of {top_pair.symbol_a}-{top_pair.symbol_b}")
    print("=" * 60)
    
    # Calculate spread and z-score over time
    prices_a = price_data[top_pair.symbol_a]
    prices_b = price_data[top_pair.symbol_b]
    
    # Calculate spread
    spread = prices_a - top_pair.beta * prices_b
    
    # Calculate rolling z-score
    rolling_mean = spread.rolling(pairs_strategy.signal_window).mean()
    rolling_std = spread.rolling(pairs_strategy.signal_window).std()
    zscore = (spread - rolling_mean) / rolling_std
    
    # Plot spread analysis
    fig, axes = plt.subplots(3, 1, figsize=(15, 10))
    fig.suptitle(f'Pairs Trading Analysis: {top_pair.symbol_a}-{top_pair.symbol_b}', fontsize=16, fontweight='bold')
    
    # Plot 1: Normalized prices
    axes[0].plot(prices_a / prices_a.iloc[0], label=top_pair.symbol_a, linewidth=2)
    axes[0].plot(prices_b / prices_b.iloc[0], label=top_pair.symbol_b, linewidth=2)
    axes[0].set_title('Normalized Prices')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: Spread
    axes[1].plot(spread, color='red', linewidth=2, label='Spread')
    axes[1].plot(rolling_mean, color='black', linestyle='--', alpha=0.7, label='Rolling Mean')
    axes[1].fill_between(spread.index, 
                        rolling_mean - 2*rolling_std, 
                        rolling_mean + 2*rolling_std, 
                        alpha=0.2, color='green', label='±2σ Band')
    axes[1].set_title(f'Spread ({top_pair.symbol_a} - {top_pair.beta:.3f} * {top_pair.symbol_b})')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    # Plot 3: Z-score with trading signals
    axes[2].plot(zscore, color='blue', linewidth=2, label='Z-Score')
    axes[2].axhline(y=0, color='black', linestyle='-', alpha=0.5)
    axes[2].axhline(y=2, color='red', linestyle='--', alpha=0.7, label='Entry (+2σ)')
    axes[2].axhline(y=-2, color='red', linestyle='--', alpha=0.7, label='Entry (-2σ)')
    axes[2].axhline(y=3, color='orange', linestyle='--', alpha=0.7, label='Stop Loss (±3σ)')
    axes[2].axhline(y=-3, color='orange', linestyle='--', alpha=0.7)
    
    # Highlight entry points
    entry_long = zscore <= -2
    entry_short = zscore >= 2
    
    axes[2].scatter(zscore[entry_long].index, zscore[entry_long], 
                   color='green', marker='^', s=50, label='Long Entry', zorder=5)
    axes[2].scatter(zscore[entry_short].index, zscore[entry_short], 
                   color='red', marker='v', s=50, label='Short Entry', zorder=5)
    
    axes[2].set_title('Z-Score and Trading Signals')
    axes[2].legend()
    axes[2].grid(True, alpha=0.3)
    axes[2].set_ylabel('Z-Score')
    
    plt.tight_layout()
    plt.show()
    
    # Print statistics
    print(f"\nSpread Statistics:")
    print(f"  Beta (hedge ratio): {top_pair.beta:.4f}")
    print(f"  Correlation: {top_pair.correlation:.3f}")
    print(f"  Cointegration p-value: {top_pair.p_value:.4f}")
    print(f"  Half-life: {top_pair.half_life:.1f} days")
    print(f"  Current Z-score: {zscore.iloc[-1]:.2f}")
    
    # Trading signal analysis
    current_zscore = zscore.iloc[-1]
    if abs(current_zscore) >= 2.0:
        signal_type = "SHORT SPREAD" if current_zscore >= 2.0 else "LONG SPREAD"
        action_a = "SELL" if current_zscore >= 2.0 else "BUY"
        action_b = "BUY" if current_zscore >= 2.0 else "SELL"
        print(f"\n🚨 TRADING SIGNAL: {signal_type}")
        print(f"  Action: {action_a} {top_pair.symbol_a}, {action_b} {top_pair.symbol_b}")
    elif abs(current_zscore) <= 0.5:
        print(f"\n✅ SPREAD NEAR EQUILIBRIUM: Consider closing positions")
    else:
        print(f"\n⏳ NO SIGNAL: Z-score {current_zscore:.2f} within normal range")
        
else:
    print("No pairs found to analyze.")

## 7. Backtest Pairs Trading Strategy

In [ ]:
# Configure backtest
backtest_config = BacktestConfig(
    start_date='2023-01-01',
    end_date='2024-06-30',
    initial_capital=100000.0,
    commission_per_share=0.005,
    market_impact=0.001,
    short_borrow_rate=0.02,
    max_positions=2,
    position_size=20000.0,
    lookback_period=100,  # Shorter for demo
    signal_window=20,
    entry_zscore=2.0,
    exit_zscore=0.0,
    stop_loss_zscore=3.0
)

print("Running pairs trading backtest...")
print(f"Period: {backtest_config.start_date} to {backtest_config.end_date}")
print(f"Initial capital: ${backtest_config.initial_capital:,.0f}")
print(f"Max positions: {backtest_config.max_positions}")

# Run backtest
try:
    results = run_pairs_backtest(price_data, backtest_config)
    
    print("\n✅ Backtest completed successfully!")
    print(f"\nBacktest Results Summary:")
    print("=" * 40)
    print(f"Total Return: {results.total_return:.2%}")
    print(f"Annual Return: {results.annual_return:.2%}")
    print(f"Volatility: {results.volatility:.2%}")
    print(f"Sharpe Ratio: {results.sharpe_ratio:.2f}")
    print(f"Max Drawdown: {results.max_drawdown:.2%}")
    print(f"\nTrading Statistics:")
    print(f"Total Trades: {results.total_trades}")
    print(f"Win Rate: {results.win_rate:.1%}")
    print(f"Profit Factor: {results.profit_factor:.2f}")
    print(f"Avg Holding Period: {results.avg_holding_period:.1f} days")
    
except Exception as e:
    print(f"❌ Backtest failed: {e}")
    results = None

## 8. Visualize Backtest Results

In [ ]:
if results and not results.equity_curve.empty:
    # Plot backtest results
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Pairs Trading Backtest Results', fontsize=16, fontweight='bold')
    
    # Plot 1: Equity curve
    axes[0, 0].plot(results.equity_curve, linewidth=2, color='blue')
    axes[0, 0].set_title(f'Portfolio Equity Curve\nTotal Return: {results.total_return:.2%}')
    axes[0, 0].set_ylabel('Portfolio Value ($)')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].axhline(y=backtest_config.initial_capital, color='red', linestyle='--', alpha=0.7, label='Initial Capital')
    axes[0, 0].legend()
    
    # Plot 2: Drawdown
    axes[0, 1].fill_between(results.drawdown_series.index, results.drawdown_series, 0, 
                           color='red', alpha=0.3)
    axes[0, 1].plot(results.drawdown_series, color='red', linewidth=2)
    axes[0, 1].set_title(f'Drawdown\nMax DD: {results.max_drawdown:.2%}')
    axes[0, 1].set_ylabel('Drawdown (%)')
    axes[0, 1].grid(True, alpha=0.3)
    
    # Plot 3: Returns distribution
    returns = results.equity_curve.pct_change().dropna()
    axes[1, 0].hist(returns, bins=30, alpha=0.7, edgecolor='black')
    axes[1, 0].axvline(returns.mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {returns.mean():.4f}')
    axes[1, 0].set_title('Daily Returns Distribution')
    axes[1, 0].set_xlabel('Daily Return')
    axes[1, 0].set_ylabel('Frequency')
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Plot 4: Performance metrics
    metrics = {
        'Sharpe Ratio': results.sharpe_ratio,
        'Calmar Ratio': results.calmar_ratio,
        'Win Rate': results.win_rate,
        'Profit Factor': min(results.profit_factor, 5.0)  # Cap for visualization
    }
    
    bars = axes[1, 1].bar(metrics.keys(), metrics.values(), 
                         color=['blue', 'green', 'orange', 'purple'], alpha=0.7)
    axes[1, 1].set_title('Performance Metrics')
    axes[1, 1].set_ylabel('Value')
    axes[1, 1].grid(True, alpha=0.3)
    
    # Add value labels on bars
    for bar, value in zip(bars, metrics.values()):
        height = bar.get_height()
        axes[1, 1].text(bar.get_x() + bar.get_width()/2., height + 0.01,
                        f'{value:.2f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    # Show trade details if available
    if results.trades:
        print(f"\nTrade Details (First 5 trades):")
        print("=" * 80)
        print(f"{'Pair':<15} {'Entry Date':<12} {'Exit Date':<12} {'Days':<6} {'P&L':<10} {'Return':<8}")
        print("=" * 80)
        
        for i, trade in enumerate(results.trades[:5]):
            pair_name = f"{trade.symbol_a}-{trade.symbol_b}"
            trade_return = trade.pnl / (abs(trade.quantity_a * trade.entry_price_a) + 
                                       abs(trade.quantity_b * trade.entry_price_b))
            print(f"{pair_name:<15} {trade.entry_date.strftime('%Y-%m-%d'):<12} "
                 f"{trade.exit_date.strftime('%Y-%m-%d'):<12} {trade.duration_days:<6} "
                 f"${trade.pnl:<9.0f} {trade_return:<7.2%}")
        
        if len(results.trades) > 5:
            print(f"... and {len(results.trades) - 5} more trades")
            
else:
    print("No backtest results to visualize.")

## 9. Generate Comprehensive Report

In [ ]:
if results:
    # Generate comprehensive backtest report
    report = generate_backtest_report(results)
    print(report)
    
    # Save report to file
    with open('pairs_trading_backtest_report.txt', 'w') as f:
        f.write(report)
    print("\n📄 Report saved to 'pairs_trading_backtest_report.txt'")
    
else:
    print("No results available for report generation.")

## 10. Integration with Trading System

The pairs trading strategy is fully integrated into the existing ML4T trading system:

### Command Line Integration
```bash
# In the trading system CLI:
trading> pairs scan                    # Scan for pairs opportunities
trading> pairs status                  # Check pairs portfolio status  
trading> pairs AAPL MSFT              # Analyze specific pair
```

### Workflow Integration
- **Pairs Analysis Agent**: Identifies cointegrated pairs during market analysis
- **Pairs Monitoring Agent**: Monitors active pairs positions and generates signals
- **Risk Integration**: Pairs positions considered in overall portfolio risk assessment
- **Signal Generation**: Pairs signals integrated into the unified signal queue

### Configuration
Pairs trading parameters can be configured through the system settings:
- Maximum number of active pairs
- Position sizing per leg
- Entry/exit thresholds
- Risk management parameters

### Safety Features
- Paper trading mode (hardcoded safety)
- Position size limits
- Stop-loss mechanisms
- Portfolio exposure limits
- Regulatory compliance integration

## Summary

This notebook demonstrates a comprehensive pairs trading implementation that includes:

✅ **Statistical Foundation**: Robust cointegration testing using Engle-Granger methodology

✅ **Signal Generation**: Z-score based entry/exit signals with configurable thresholds

✅ **Risk Management**: Stop-loss controls, position sizing, and portfolio limits

✅ **Backtesting Framework**: Comprehensive performance evaluation with realistic costs

✅ **System Integration**: Seamless integration with existing LangGraph-based trading workflow

✅ **Monitoring Tools**: Real-time spread monitoring and position management

The implementation follows best practices for statistical arbitrage and provides a solid foundation for pairs trading in a systematic environment.

### Next Steps
1. **Live Testing**: Test with paper trading using real market data
2. **Parameter Optimization**: Fine-tune entry/exit thresholds based on historical performance
3. **Advanced Features**: Add multi-timeframe analysis and dynamic hedge ratio estimation
4. **Risk Enhancement**: Implement correlation decay detection and pair quality scoring
5. **Portfolio Integration**: Optimize pairs allocation within overall portfolio strategy